In [1]:
# NB24 CELL 1
# Purpose: Load primary model and all components needed for external validation
# Validate TWO new cohorts: GSE31210 and TCGA-LUSC
# Also re-validate GSE72094 and CPTAC with stacking ensemble
# New notebook: NB24_external_validation.ipynb

import os
import pickle
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

os.chdir("/Users/parthshringarpure/Desktop/AI/Projects/luad_survival")

# --- Load primary model ---
with open("models/final_leakagefree/xgboost_final.pkl", "rb") as f:
    model_xgb = pickle.load(f)
with open("models/final_leakagefree/scaler_xgb.pkl", "rb") as f:
    scaler_xgb = pickle.load(f)

feature_names = list(scaler_xgb.feature_names_in_)
print(f"Primary model loaded: {len(feature_names)} features")

# --- Load GTEx reference for dysregulation ---
with open("data/processed/gtex_reference.json") as f:
    gtex_ref = json.load(f)
print(f"GTEx reference: {len(gtex_ref)} genes")

# --- Load LM22 for CIBERSORT ---
lm22 = pd.read_csv("data/external/LM22.txt", sep="\t", index_col=0)
lm22_clipped = np.clip(lm22.values, 0, 20)
lm22_linear  = (2 ** lm22_clipped) - 1
lm22_linear  = np.clip(lm22_linear, 0, 1e6)

from sklearn.preprocessing import normalize
lm22_norm = normalize(lm22_linear, axis=0)
lm22_genes = lm22.index.tolist()
print(f"LM22 loaded: {lm22.shape}")

# --- Parse feature groups from scaler ---
expr_feat_genes   = [f.replace('_expr',   '') for f in feature_names
                     if f.endswith('_expr')]
dysreg_feat_genes = [f.replace('_dysreg', '') for f in feature_names
                     if f.endswith('_dysreg')]
immune_feat_names = [f for f in feature_names
                     if not f.endswith('_expr')
                     and not f.endswith('_dysreg')
                     and f not in ['age', 'gender',
                                   'stage_Stage II',
                                   'stage_Stage III',
                                   'stage_Stage IV',
                                   'stageIII_x_M2',
                                   'stageIV_x_CD8',
                                   'age_x_stageIII',
                                   'stageIII_x_Treg',
                                   'M2_x_CD8']]

print(f"\nFeature groups:")
print(f"  Expression genes:  {len(expr_feat_genes)}")
print(f"  Dysreg genes:      {len(dysreg_feat_genes)}")
print(f"  Immune cell types: {len(immune_feat_names)}")

# --- DIY CIBERSORT function (from NB06) ---
from sklearn.svm import NuSVR
from scipy.optimize import nnls

def run_cibersort_patient(expr_log2_values, lm22_gene_mask):
    expr_clipped = np.clip(expr_log2_values, 0, 20)
    expr_linear  = (2 ** expr_clipped) - 1
    expr_linear  = np.clip(expr_linear, 0, 1e6)
    lm22_sub     = lm22_norm[lm22_gene_mask]
    expr_norm    = normalize(expr_linear.reshape(1, -1))[0]
    try:
        svr = NuSVR(nu=0.5, kernel='linear', C=1.0, max_iter=500)
        svr.fit(lm22_sub, expr_norm)
        raw = svr.coef_[0]
    except Exception:
        raw = np.zeros(22)
    clipped = np.maximum(raw, 0)
    if clipped.sum() == 0:
        try:
            clipped, _ = nnls(lm22_sub, expr_norm)
        except Exception:
            clipped = np.ones(22) / 22
    total = clipped.sum()
    return clipped / total if total > 0 else np.ones(22) / 22

# --- Inference pipeline function ---
def build_feature_matrix(expr_df, clin_df,
                         dysreg_df=None,
                         immune_df=None,
                         platform='rnaseq'):
    """
    Build 124-feature matrix for primary model inference.
    expr_df:   patients x genes (log2 normalised)
    clin_df:   patients x clinical (age, gender, stage_clean)
    dysreg_df: patients x dysreg genes (optional — computed if None)
    immune_df: patients x 22 immune (optional — computed if None)
    platform:  'rnaseq' or 'microarray'
    """
    patients = expr_df.index.tolist()
    n = len(patients)

    # --- Expression block ---
    # Fill missing genes with 0
    expr_block = pd.DataFrame(index=patients)
    for g in expr_feat_genes:
        if g in expr_df.columns:
            expr_block[f"{g}_expr"] = expr_df[g].values
        else:
            expr_block[f"{g}_expr"] = 0.0

    # --- Dysregulation block ---
    if dysreg_df is not None:
        dysreg_block = pd.DataFrame(index=patients)
        for g in dysreg_feat_genes:
            if g in dysreg_df.columns:
                dysreg_block[f"{g}_dysreg"] = dysreg_df[g].values
            else:
                dysreg_block[f"{g}_dysreg"] = 0.0
    else:
        # Compute from GTEx reference
        dysreg_block = pd.DataFrame(index=patients)
        for g in dysreg_feat_genes:
            if g in gtex_ref and g in expr_df.columns:
                z = ((expr_df[g].values - gtex_ref[g]['mean'])
                     / max(gtex_ref[g]['std'], 1e-6))
                dysreg_block[f"{g}_dysreg"] = z
            else:
                dysreg_block[f"{g}_dysreg"] = 0.0

    # ---

Primary model loaded: 124 features
GTEx reference: 819 genes
LM22 loaded: (547, 22)

Feature groups:
  Expression genes:  72
  Dysreg genes:      20
  Immune cell types: 22


In [2]:
# NB24 CELL 1 (fix)
# Purpose: Verify immune_feat_names and OOF file, patch if needed

import os

# --- Check immune_feat_names ---
print(f"immune_feat_names count: {len(immune_feat_names)}")
print(f"First 5: {immune_feat_names[:5]}")

# If empty, rebuild from feature_names directly
if len(immune_feat_names) == 0:
    skip = set(
        [f for f in feature_names if f.endswith('_expr')] +
        [f for f in feature_names if f.endswith('_dysreg')] +
        ['age', 'gender',
         'stage_Stage II', 'stage_Stage III', 'stage_Stage IV',
         'stageIII_x_M2', 'stageIV_x_CD8', 'age_x_stageIII',
         'stageIII_x_Treg', 'M2_x_CD8']
    )
    immune_feat_names = [f for f in feature_names if f not in skip]
    print(f"Rebuilt immune_feat_names: {len(immune_feat_names)}")
    print(f"Names: {immune_feat_names}")

# --- Check OOF file ---
oof_path = "outputs/results/oof_predictions_nb22.csv"
if os.path.exists(oof_path):
    oof_df = pd.read_csv(oof_path, index_col=0)
    print(f"\nOOF predictions loaded: {oof_df.shape}")
    print(f"Columns: {list(oof_df.columns)}")
else:
    print(f"\nOOF file not found at {oof_path}")
    print("Contents of outputs/results/:")
    print(os.listdir("outputs/results/"))

print("\nInference pipeline function: defined ✓")
print("=== CELL 1 (fix) COMPLETE ===")
print("Next: Cell 2 — Download and load GSE31210")

immune_feat_names count: 22
First 5: ['B cells naive', 'B cells memory', 'Plasma cells', 'T cells CD8', 'T cells CD4 naive']

OOF predictions loaded: (920, 4)
Columns: ['oof_bl1', 'oof_bl2', 'oof_bl3', 'oof_meta']

Inference pipeline function: defined ✓
=== CELL 1 (fix) COMPLETE ===
Next: Cell 2 — Download and load GSE31210


In [3]:
# NB24 CELL 2
# Purpose: Download GSE31210 from GEO and extract expression + survival
# 226 LUAD patients, GPL570 microarray, Japanese cohort
# Same pipeline as NB15 (GSE68465) — GEOparse + probe-to-gene map

import GEOparse
import pandas as pd
import numpy as np
import os

# --- Download GSE31210 ---
# GEOparse will download to data/external/ if not already present
gse31210_path = "data/external/GSE31210_family.soft.gz"

if os.path.exists(gse31210_path):
    print(f"GSE31210 already downloaded: {gse31210_path}")
else:
    print("Downloading GSE31210 from GEO (this may take 2-5 minutes)...")
    gse = GEOparse.get_GEO(
        geo="GSE31210",
        destdir="data/external/",
        silent=False
    )
    print("Download complete.")

print("Loading GSE31210...")
gse = GEOparse.get_GEO(
    filepath=gse31210_path,
    silent=True
)
print(f"Samples loaded: {len(gse.gsms)}")

# --- Inspect platform ---
platform_id = list(gse.gpls.keys())[0]
gpl = gse.gpls[platform_id]
print(f"Platform: {platform_id}")
print(f"GPL columns: {list(gpl.table.columns)}")
print(f"GPL shape: {gpl.table.shape}")

# --- Inspect first sample ---
first_gsm = list(gse.gsms.values())[0]
print(f"\nFirst GSM columns: {list(first_gsm.table.columns)}")
print(f"First 3 probe IDs: {first_gsm.table.iloc[:3, 0].tolist()}")

# --- Inspect clinical metadata ---
clinical_rows = []
for gsm_id, gsm in gse.gsms.items():
    meta = gsm.metadata
    row  = {'sample_id': gsm_id}
    for char in meta.get('characteristics_ch1', []):
        if ':' in char:
            key, val = char.split(':', 1)
            row[key.strip().lower().replace(' ', '_')] = val.strip()
    clinical_rows.append(row)

clinical_df = pd.DataFrame(clinical_rows).set_index('sample_id')
print(f"\nClinical DataFrame shape: {clinical_df.shape}")
print(f"Columns: {list(clinical_df.columns)}")
print(f"\nFirst row sample:\n{clinical_df.iloc[0]}")

19-Jun-2026 17:23:52 DEBUG utils - Directory data/external/ already exists. Skipping.
19-Jun-2026 17:23:52 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE31nnn/GSE31210/soft/GSE31210_family.soft.gz to data/external/GSE31210_family.soft.gz


100%|██████████| 109M/109M [00:06<00:00, 18.8MB/s]   
19-Jun-2026 17:23:59 DEBUG downloader - Size validation passed
19-Jun-2026 17:23:59 DEBUG downloader - Moving /var/folders/9z/l9kb5kss3jbdmdjwjzchmxj40000gn/T/tmp2owjl4q7 to /Users/parthshringarpure/Desktop/AI/Projects/luad_survival/data/external/GSE31210_family.soft.gz
19-Jun-2026 17:23:59 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE31nnn/GSE31210/soft/GSE31210_family.soft.gz
19-Jun-2026 17:23:59 INFO GEOparse - Parsing data/external/GSE31210_family.soft.gz: 
19-Jun-2026 17:23:59 DEBUG GEOparse - DATABASE: GeoMiame
19-Jun-2026 17:23:59 DEBUG GEOparse - SERIES: GSE31210
19-Jun-2026 17:23:59 DEBUG GEOparse - PLATFORM: GPL570
19-Jun-2026 17:24:00 DEBUG GEOparse - SAMPLE: GSM773540
19-Jun-2026 17:24:00 DEBUG GEOparse - SAMPLE: GSM773541
19-Jun-2026 17:24:00 DEBUG GEOparse - SAMPLE: GSM773542
19-Jun-2026 17:24:00 DEBUG GEOparse - SAMPLE: GSM773543
19-Jun-2026 17:24:00 DEBUG GEOparse - SAMPLE: GSM7

Download complete.
Loading GSE31210...
Samples loaded: 246
Platform: GPL570
GPL columns: ['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID', 'RefSeq Transcript ID', 'Gene Ontology Biological Process', 'Gene Ontology Cellular Component', 'Gene Ontology Molecular Function']
GPL shape: (54675, 16)

First GSM columns: ['ID_REF', 'VALUE']
First 3 probe IDs: ['1007_s_at', '1053_at', '117_at']

Clinical DataFrame shape: (246, 19)
Columns: ['tissue', 'age_(years)', 'gender', 'smoking_status', 'bi', 'pathological_stage', 'pstage_iorii', 'gene_alteration_status', 'myc', 'myc_copy', 'cluster', 'relapse', 'days_before_relapse/censor', 'months_before_relapse/censor', 'death', 'days_before_death/censor', 'exclude_for_prognosis_analysis_due_to_incomplete_resection_or_adjuvant_therapy', 'group', 'age']

First row sample:
tissue                      

In [4]:
# NB24 CELL 3
# Purpose: Extract expression matrix and survival from GSE31210
# Filter: exclude patients with incomplete resection or adjuvant therapy
# Expected: ~226 patients after exclusion (246 - ~20 excluded)

import pandas as pd
import numpy as np
import GEOparse

# --- Build probe-to-gene map from GPL570 ---
gpl_table = gpl.table[['ID', 'Gene Symbol']].copy()
gpl_table.columns = ['probe_id', 'gene_symbol']
gpl_table = gpl_table[gpl_table['gene_symbol'].notna()]
gpl_table = gpl_table[gpl_table['gene_symbol'] != '']
gpl_table = gpl_table[~gpl_table['gene_symbol'].str.contains('///', na=False)]
gpl_table = gpl_table.set_index('probe_id')
print(f"GPL570 probes with unambiguous gene symbol: {len(gpl_table)}")

# --- Extract expression values ---
print("Extracting expression values...")
expr_dict = {}
for gsm_id, gsm in gse.gsms.items():
    table = gsm.table[['ID_REF', 'VALUE']].copy()
    table = table.set_index('ID_REF')['VALUE']
    table.index = table.index.map(
        lambda p: gpl_table.loc[p, 'gene_symbol']
        if p in gpl_table.index else None
    )
    table = table[table.index.notna()]
    expr_dict[gsm_id] = table

print("Building expression matrix...")
expr_raw = pd.DataFrame(expr_dict).T
print(f"Shape after probe extraction: {expr_raw.shape}")

# Average duplicate gene symbols
expr_raw  = expr_raw.astype(float)
expr_avg  = expr_raw.groupby(level=0, axis=1).mean()
print(f"Shape after averaging duplicates: {expr_avg.shape}")

# Check scale and log2 transform
raw_max = expr_avg.values.max()
raw_min = expr_avg.values.min()
print(f"Pre-log2 range: {raw_min:.2f} to {raw_max:.2f}")
print(f"Linear scale: {'YES' if raw_max > 100 else 'NO — may already be log2'}")

if raw_max > 100:
    expr_log2 = np.log2(expr_avg + 1)
else:
    expr_log2 = expr_avg.copy()
print(f"Post-log2 range: {expr_log2.values.min():.3f} "
      f"to {expr_log2.values.max():.3f}")

# --- Parse clinical / survival ---
clin = clinical_df.copy()

# Survival time
clin['survival_time'] = pd.to_numeric(
    clin['days_before_death/censor'], errors='coerce'
)
# Vital status
clin['event'] = (
    clin['death'].str.strip().str.lower() == 'dead'
).astype(int)

# Age
clin['age'] = pd.to_numeric(
    clin['age_(years)'], errors='coerce'
)

# Gender
clin['gender'] = clin['gender'].str.strip().str.lower()

# Stage normalisation
def normalise_stage_gse31210(s):
    if not isinstance(s, str):
        return None
    s = s.strip().upper()
    if s in ['IA', 'IB', 'I']:    return 'Stage I'
    elif s in ['IIA', 'IIB', 'II']: return 'Stage II'
    elif s in ['IIIA', 'IIIB', 'III']: return 'Stage III'
    elif s in ['IV']:               return 'Stage IV'
    else:                           return None

clin['stage_clean'] = clin['pathological_stage'].apply(
    normalise_stage_gse31210
)

print(f"\nStage distribution (before exclusion):")
print(clin['stage_clean'].value_counts(dropna=False))

# --- Apply exclusion filter ---
exclude_col = 'exclude_for_prognosis_analysis_due_to_incomplete_resection_or_adjuvant_therapy'
excluded = clin[exclude_col].str.strip().str.lower() == 'exclude'
print(f"\nExcluded patients: {excluded.sum()}")
clin_clean = clin[~excluded].copy()
print(f"After exclusion: {len(clin_clean)} patients")

# --- Drop missing survival ---
clin_clean = clin_clean[
    clin_clean['survival_time'].notna() &
    (clin_clean['survival_time'] > 0)
].copy()
print(f"After dropping missing survival: {len(clin_clean)} patients")

# --- Merge with expression ---
common = clin_clean.index.intersection(expr_log2.index)
expr_31210  = expr_log2.loc[common].copy()
clin_31210  = clin_clean.loc[common].copy()

print(f"\nFinal GSE31210:")
print(f"  Patients: {len(common)}")
print(f"  Events:   {clin_31210['event'].sum()} "
      f"({100*clin_31210['event'].mean():.1f}%)")
print(f"  Survival: {clin_31210['survival_time'].min():.0f} "
      f"to {clin_31210['survival_time'].max():.0f} days")
print(f"  Stage:\n{clin_31210['stage_clean'].value_counts(dropna=False)}")
print(f"  Expression: {expr_31210.shape}")

print("\n=== CELL 3 COMPLETE ===")
print("Next: Cell 4 — Build features and validate primary model on GSE31210")

GPL570 probes with unambiguous gene symbol: 42986
Extracting expression values...
Building expression matrix...
Shape after probe extraction: (246, 42986)
Shape after averaging duplicates: (246, 21655)
Pre-log2 range: 0.06 to 49858.57
Linear scale: YES
Post-log2 range: 0.085 to 15.606

Stage distribution (before exclusion):
stage_clean
Stage I     168
Stage II     58
None         20
Name: count, dtype: int64

Excluded patients: 22
After exclusion: 224 patients
After dropping missing survival: 204 patients

Final GSE31210:
  Patients: 204
  Events:   30 (14.7%)
  Survival: 221 to 3863 days
  Stage:
stage_clean
Stage I     162
Stage II     42
Name: count, dtype: int64
  Expression: (204, 21655)

=== CELL 3 COMPLETE ===
Next: Cell 4 — Build features and validate primary model on GSE31210


In [7]:
# NB24 CELL 4 (fix)
# Build feature matrix manually step by step for GSE31210
# Avoids silent failure in build_feature_matrix function

import numpy as np
import pandas as pd
from sksurv.metrics import concordance_index_censored
import warnings
warnings.filterwarnings('ignore')

patients = expr_31210.index.tolist()
n = len(patients)
print(f"Building features for {n} patients...")

# --- Block 1: Expression (72 Lasso genes) ---
expr_block = pd.DataFrame(index=patients)
expr_missing = []
for g in expr_feat_genes:
    if g in expr_31210.columns:
        expr_block[f"{g}_expr"] = expr_31210[g].values
    else:
        expr_block[f"{g}_expr"] = 0.0
        expr_missing.append(g)
print(f"Expression block: {expr_block.shape}")
print(f"Missing genes (filled 0): {len(expr_missing)}")
if expr_missing:
    print(f"  {expr_missing[:5]}...")

# --- Block 2: Dysregulation (20 genes) ---
dysreg_block = pd.DataFrame(index=patients)
dysreg_missing = []
for g in dysreg_feat_genes:
    if g in expr_31210.columns and g in gtex_ref:
        z = ((expr_31210[g].values - gtex_ref[g]['mean'])
             / max(gtex_ref[g]['std'], 1e-6))
        dysreg_block[f"{g}_dysreg"] = z
    else:
        dysreg_block[f"{g}_dysreg"] = 0.0
        dysreg_missing.append(g)
print(f"Dysreg block: {dysreg_block.shape}")
print(f"Missing dysreg (filled 0): {len(dysreg_missing)}")

# --- Block 3: CIBERSORT immune (22 cell types) ---
print(f"\nRunning CIBERSORT on {n} patients...")
lm22_in_31210 = [g for g in lm22_genes if g in expr_31210.columns]
lm22_idx      = [i for i, g in enumerate(lm22_genes)
                 if g in expr_31210.columns]
lm22_norm_sub = lm22_norm[lm22_idx]  # subset rows to available genes
print(f"LM22 genes available: {len(lm22_in_31210)}/547")

immune_vals = []
for i, pid in enumerate(patients):
    expr_vals = expr_31210.loc[pid, lm22_in_31210].values.astype(float)
    # Back-transform log2 to linear
    expr_clipped = np.clip(expr_vals, 0, 20)
    expr_linear  = (2 ** expr_clipped) - 1
    expr_linear  = np.clip(expr_linear, 0, 1e6)
    expr_norm_p  = normalize(expr_linear.reshape(1, -1))[0]
    # NuSVR
    try:
        from sklearn.svm import NuSVR
        svr = NuSVR(nu=0.5, kernel='linear', C=1.0, max_iter=500)
        svr.fit(lm22_norm_sub, expr_norm_p)
        raw = svr.coef_[0]
    except Exception:
        raw = np.zeros(22)
    clipped = np.maximum(raw, 0)
    if clipped.sum() == 0:
        from scipy.optimize import nnls
        try:
            clipped, _ = nnls(lm22_norm_sub, expr_norm_p)
        except Exception:
            clipped = np.ones(22) / 22
    total = clipped.sum()
    fracs = clipped / total if total > 0 else np.ones(22) / 22
    immune_vals.append(fracs)
    if (i + 1) % 50 == 0:
        print(f"  CIBERSORT: {i+1}/{n}...")

immune_block = pd.DataFrame(
    immune_vals, index=patients, columns=lm22.columns
)
print(f"Immune block: {immune_block.shape}")
row_sums = immune_block.sum(axis=1)
print(f"Row sums: {row_sums.min():.4f} to {row_sums.max():.4f}")

# --- Block 4: Clinical + interactions ---
age_vals   = clin_31210['age'].fillna(clin_31210['age'].median()).values
gender_vals = (clin_31210['gender'].str.lower()
               .isin(['male', 'm'])
               .astype(float).values)
stage_II  = (clin_31210['stage_clean'] == 'Stage II').astype(float).values
stage_III = (clin_31210['stage_clean'] == 'Stage III').astype(float).values
stage_IV  = (clin_31210['stage_clean'] == 'Stage IV').astype(float).values

m2   = immune_block['Macrophages M2'].values
cd8  = immune_block['T cells CD8'].values
treg = immune_block['T cells regulatory (Tregs)'].values

clin_block = pd.DataFrame({
    'age'             : age_vals,
    'gender'          : gender_vals,
    'stage_Stage II'  : stage_II,
    'stage_Stage III' : stage_III,
    'stage_Stage IV'  : stage_IV,
    'stageIII_x_M2'   : stage_III * m2,
    'stageIV_x_CD8'   : stage_IV  * cd8,
    'age_x_stageIII'  : age_vals  * stage_III,
    'stageIII_x_Treg' : stage_III * treg,
    'M2_x_CD8'        : m2        * cd8,
}, index=patients)

# --- Assemble in exact feature order ---
X_31210 = pd.concat([
    expr_block,
    dysreg_block,
    immune_block[immune_feat_names],
    clin_block
], axis=1)[feature_names]

print(f"\nFinal feature matrix: {X_31210.shape}")
print(f"Column order correct: {list(X_31210.columns) == feature_names}")
print(f"NaNs: {X_31210.isna().sum().sum()}")

# --- Scale and predict ---
X_scaled   = scaler_xgb.transform(X_31210)
risk_31210 = model_xgb.predict(
    pd.DataFrame(X_scaled, columns=feature_names, index=X_31210.index)
)

# --- C-index ---
y_time_31210  = clin_31210['survival_time'].values.astype(float)
y_event_31210 = clin_31210['event'].values.astype(bool)

ci_31210 = concordance_index_censored(
    y_event_31210, y_time_31210, risk_31210
)[0]

print(f"\n=== GSE31210 PRIMARY MODEL RESULT ===")
print(f"C-index: {ci_31210:.4f}")
print(f"N={len(clin_31210)}, Events={y_event_31210.sum()} "
      f"({100*y_event_31210.mean():.1f}%)")

# --- Bootstrap 95% CI ---
np.random.seed(42)
boot_ci = []
n_pts = len(y_time_31210)
for _ in range(1000):
    idx = np.random.choice(n_pts, n_pts, replace=True)
    if y_event_31210[idx].sum() < 2:
        continue
    try:
        ci_b = concordance_index_censored(
            y_event_31210[idx], y_time_31210[idx], risk_31210[idx]
        )[0]
        boot_ci.append(ci_b)
    except Exception:
        continue

ci_lower = np.percentile(boot_ci, 2.5)
ci_upper = np.percentile(boot_ci, 97.5)
print(f"95% Bootstrap CI: ({ci_lower:.4f}, {ci_upper:.4f})")

print("\n=== CELL 4 (fix) COMPLETE ===")
print("Next: Cell 5 — Validate stacking ensemble on GSE31210")

Building features for 204 patients...
Expression block: (204, 72)
Missing genes (filled 0): 11
  ['EPGN', 'LST-3TM12', 'C8orf47', 'TMEM215', 'MT1A']...
Dysreg block: (204, 20)
Missing dysreg (filled 0): 2

Running CIBERSORT on 204 patients...
LM22 genes available: 520/547
  CIBERSORT: 50/204...
  CIBERSORT: 100/204...
  CIBERSORT: 150/204...
  CIBERSORT: 200/204...
Immune block: (204, 22)
Row sums: 1.0000 to 1.0000

Final feature matrix: (204, 124)
Column order correct: True
NaNs: 0

=== GSE31210 PRIMARY MODEL RESULT ===
C-index: 0.5230
N=204, Events=30 (14.7%)
95% Bootstrap CI: (0.4139, 0.6373)

=== CELL 4 (fix) COMPLETE ===
Next: Cell 5 — Validate stacking ensemble on GSE31210


In [8]:
# NB24 CELL 5
# Purpose: Validate stacking ensemble on GSE31210
# Also re-validate stacking ensemble on GSE72094 and CPTAC
# Build the complete comparison table:
# | Cohort | Primary (0.702) | Stacking (0.687) |

import numpy as np
import pandas as pd
from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

# --- Stacking ensemble needs to be refit on TCGA+GSE68465 training data ---
# Load training data from NB22
features_base = pd.read_csv(
    "data/processed/features_base_920.csv", index_col=0
)
features_dysreg = pd.read_csv(
    "data/processed/features_dysreg_920.csv", index_col=0
)
survival_920 = pd.read_csv(
    "data/processed/survival_920.csv", index_id=0
)

TypeError: read_csv() got an unexpected keyword argument 'index_id'

In [10]:
import pandas as pd
import numpy as np
import pickle
import json
import GEOparse
import warnings
warnings.filterwarnings('ignore')

base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'

# ── Load TCGA genes ───────────────────────────────────────────────
expr_full = pd.read_csv(f'{base}/data/processed/expression_full_478.csv', 
                         index_col=0)
tcga_genes = set(expr_full.columns)
print(f"TCGA genes:     {len(tcga_genes)}")

# ── Load GSE68465 genes ───────────────────────────────────────────
print("\nLoading GSE68465...")
gse68465 = GEOparse.get_GEO(geo="GSE68465",
                              destdir=f'{base}/data/external/',
                              silent=True)
gpl96 = gse68465.gpls['GPL96']
probe_to_gene_96 = gpl96.table.set_index('ID')['Gene Symbol'].dropna()
probe_to_gene_96 = probe_to_gene_96[probe_to_gene_96 != '']
gse68465_genes = set(probe_to_gene_96.values)
print(f"GSE68465 genes: {len(gse68465_genes)}")

# ── Load GSE72094 genes ───────────────────────────────────────────
print("Loading GSE72094...")
gse72094 = GEOparse.get_GEO(geo="GSE72094",
                              destdir=f'{base}/data/external/',
                              silent=True)
gpl15048 = gse72094.gpls['GPL15048']
probe_to_gene_72 = gpl15048.table.set_index('ID')['GeneSymbol'].dropna()
probe_to_gene_72 = probe_to_gene_72[probe_to_gene_72 != '']
gse72094_genes = set(probe_to_gene_72.values)
print(f"GSE72094 genes: {len(gse72094_genes)}")

# ── Three-way intersection ─────────────────────────────────────────
intersection_all = tcga_genes.intersection(
                   gse68465_genes).intersection(gse72094_genes)
print(f"\nThree-way intersection: {len(intersection_all)} genes")
print(f"Coverage vs TCGA: {len(intersection_all)/len(tcga_genes)*100:.1f}%")

# ── Check current 72 Lasso genes ─────────────────────────────────
cox_lasso   = pickle.load(open(f'{base}/models/cox_lasso_expression.pkl','rb'))
gene_list   = json.load(open(f'{base}/models/gene_list.json'))
coefs       = cox_lasso.coef_[:, 0]
lasso_genes = [g for g, s in zip(gene_list, coefs != 0) if s]

in_intersection  = [g for g in lasso_genes if g in intersection_all]
out_intersection = [g for g in lasso_genes if g not in intersection_all]

print(f"\nCurrent 72 Lasso genes in intersection: {len(in_intersection)}/72")
print(f"Missing from intersection:              {len(out_intersection)}/72")

# ── Check coefficients of missing genes ──────────────────────────
coef_dict = dict(zip(gene_list, coefs))
print(f"\nMissing genes and their Lasso coefficients:")
missing_coefs = [(g, coef_dict[g]) for g in out_intersection]
missing_coefs.sort(key=lambda x: abs(x[1]), reverse=True)
for gene, coef in missing_coefs:
    print(f"  {gene:<20} coef={coef:+.4f}")

# ── Decision threshold ────────────────────────────────────────────
print(f"\n{'='*45}")
print(f"VERDICT:")
if len(in_intersection) >= 60:
    print(f"✅ {len(in_intersection)}/72 genes in intersection")
    print(f"PROCEED — strong overlap, approach is viable")
elif len(in_intersection) >= 50:
    print(f"⚠️  {len(in_intersection)}/72 genes in intersection")
    print(f"BORDERLINE — proceed with caution")
else:
    print(f"❌ {len(in_intersection)}/72 genes in intersection")
    print(f"STOP — too many high-value genes missing")
print(f"{'='*45}")

TCGA genes:     20502

Loading GSE68465...
GSE68465 genes: 13515
Loading GSE72094...
GSE72094 genes: 22115

Three-way intersection: 11449 genes
Coverage vs TCGA: 55.8%

Current 72 Lasso genes in intersection: 42/72
Missing from intersection:              30/72

Missing genes and their Lasso coefficients:
  PKHD1L1              coef=-0.3594
  CNTN3                coef=-0.2610
  KCNA6                coef=-0.2273
  EPGN                 coef=+0.2142
  WFDC3                coef=-0.2028
  GSTT2                coef=-0.1921
  GPC6                 coef=+0.1683
  TMEM139              coef=+0.1556
  PCSK9                coef=+0.1413
  TMEM215              coef=-0.1311
  LOC654433            coef=+0.1274
  MPV17L               coef=-0.1242
  FBN3                 coef=-0.1103
  C8orf47              coef=-0.0966
  LST-3TM12            coef=+0.0944
  CYP2D7P1             coef=+0.0919
  ODZ1                 coef=-0.0860
  KNDC1                coef=-0.0838
  MT1A                 coef=+0.0725
  CASP14  